# Clase 12: Intro a GeoPandas

Una introducción a GeoPandas, una librería de Python para trabajar con datos geoespaciales. En esta clase, aprenderemos a cargar, visualizar y limpiar datos geoespaciales, así como a realizar análisis espacial utilizando dicha librería.

Desde la asignatura, esta librería nos aporta complementos a los siguientes temas:

- Detección de Outliers (geograficos)
- Imputación (en base a ubicación espacial)
- Ingenieria de atributos (en base a las distancias)
- Utilización de datos externos

In [ ]:
# Instala GeoPandas si no está disponible (común en Colab o entornos nuevos)
! pip install -Uq geopandas shapely pyproj rtree

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt

import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos

Utilizaremos tres fuentes de datos:
*   **Propiedades**: El dataset de la competencia.
*   **Barrios de CABA**: Un archivo GeoJSON con los polígonos de los barrios.
*   **Estaciones de Subte**: Un archivo CSV con las coordenadas de las estaciones.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DIR = "/content/drive/MyDrive/DM/2026/datos"
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")
df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col='id')

## 2. Limpieza y Preprocesamiento

Aplicamos aproximadamente el filtro que ya viene consensuado en la competencia

In [ ]:
df = df_ent.copy()
rows_before = len(df)

# Normalización de tipos de propiedad
df["property_type"] = df["property_type"].replace({"casas": "casa", "departamentos": "departamento"})

# Filtros de selección
mask = (
    df["price"].notna() &
    (df["currency_type"] == 'dolares') &
    df["location_1"].isin(["Capital Federal", "Ciudad Autónoma de Buenos Aires", 'CABA']) &
    (df["operation_type"] == 'venta') &
    df["property_type"].isin(["departamento", "casa", "cochera"])
)

df = df.loc[mask]
rows_after = len(df)

print(f"Filas iniciales: {rows_before}")
print(f"Filas tras filtros: {rows_after} (Se mantuvo el {rows_after/rows_before*100:.2f}% del dataset)")

## 3. Conversión a GeoDataFrame

Para operar geográficamente, transformamos el DataFrame de Pandas a un **GeoDataFrame** de GeoPandas, definiendo la columna `geometry` a partir de las columnas `lat` y `lon`.

In [ ]:
# Eliminamos registros sin coordenadas
df = df.dropna(subset=["lat", "lon"])

gdf_ent = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326" # Coordenadas geográficas WGS84
)

gdf_ent.head(3)

In [ ]:
gdf_ent.geometry.dtype

## 4. Trabajo con Polígonos: Barrios de CABA

Cargamos los polígonos de los barrios desde el portal de datos abiertos de la ciudad.

In [ ]:
URL_BARRIOS = "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/ministerio-de-educacion/barrios/barrios.geojson"
gdf_barrios = gpd.read_file(URL_BARRIOS)

print(f"Cargados {len(gdf_barrios)} barrios.")
gdf_barrios.plot(figsize=(8, 8), edgecolor='white', color='lightblue')
plt.title("Barrios de la Ciudad Autónoma de Buenos Aires")
plt.show()

### 4.1. Limpieza por contorno (Outliers geográficos)

A veces los datos vienen con errores de carga en latitud/longitud. Una técnica común es filtrar puntos que caen fuera de la "envolvente convexa" (Convex Hull) de la ciudad.

In [ ]:
gdf_barrios.head()

In [ ]:
# Disolvemos todos los barrios en un solo polígonos que representa la ciudad
caba_poly = gdf_barrios.dissolve()
caba_hull = caba_poly.convex_hull.iloc[0]

# Identificamos puntos dentro y fuera
gdf_ent['is_within'] = gdf_ent.geometry.within(caba_hull)

print(f"Puntos dentro del Hull: {gdf_ent['is_within'].sum()}")
print(f"Puntos fuera del Hull: {(~gdf_ent['is_within']).sum()}")

In [ ]:
# Visualización de la limpieza geográfica
fig, ax = plt.subplots(figsize=(10, 8))
gdf_barrios.plot(ax=ax, color='lightgray', edgecolor='white')

# Graficamos puntos de fuera en rojo y dentro en azul (usando una muestra para no saturar)
gdf_ent[~gdf_ent['is_within']].plot(ax=ax, color='red', markersize=10, label='Outliers (Fuera del Hull)')
gdf_ent[gdf_ent['is_within']].sample(2000).plot(ax=ax, color='blue', markersize=2, alpha=0.3, label='Puntos Válidos')

plt.title("Detección de Outliers Geográficos (Convex Hull CABA)")
plt.legend()
plt.show()

In [ ]:
# Aplicamos el filtro definitivo
gdf_ent = gdf_ent[gdf_ent['is_within']].copy()
gdf_ent = gdf_ent.drop(columns=['is_within'])
print(f"Filas finales después de limpieza geográfica: {gdf_ent.shape[0]}")

## 5. Spatial Join: Imputación de Barrios

Muchos registros tienen el barrio (`location_3`) como nulo o incorrecto. Utilizaremos un **Spatial Join** para asignar el barrio real basado en las coordenadas de la propiedad.

In [ ]:
# Estado inicial de location_3
missing_before = gdf_ent["location_3"].isna().sum()
print(f"Propiedades con location_3 nulo ANTES del join: {missing_before}")

In [ ]:
# Realizamos el join espacial
# how='left' mantiene todas las propiedades
# predicate='within' busca en qué barrio cae el punto
gdf_ent = gpd.sjoin(gdf_ent, gdf_barrios[['nombre', 'geometry']], how='left', predicate='within')

# Reemplazamos location_3 por el nombre del barrio obtenido del join
gdf_ent['location_3'] = gdf_ent['nombre']
gdf_ent = gdf_ent.drop(columns=['index_right', 'nombre'])

missing_after = gdf_ent["location_3"].isna().sum()
print(f"Propiedades con location_3 nulo DESPUÉS del join: {missing_after}")

print("\nTop 5 barrios con más publicaciones (imputados):")
print(gdf_ent['location_3'].value_counts().head())

## 6. Cálculo de Distancias

### 6.1. Distancia a puntos específicos (Fórmula de Haversine)
Cuando trabajamos con pocos puntos, podemos usar la fórmula manual de Haversine.

In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    """Función matematica sencilla que permite calcular la distancia a dos puntos
        en kms, dadas las coordenadas de ambos puntos en grados. Evita la
        reproyección que es costosa en espacio (ram) y tiempo (procesamiento)."""
    R = 6371 # Radio de la Tierra en km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Plaza Serrano
lat_ps, lon_ps = -34.588972, -58.430277

gdf_ent['dist_plaza_serrano'] = gdf_ent.apply(
    lambda r: haversine(r['lat'], r['lon'], lat_ps, lon_ps), axis=1
)

print(f"Distancia media a Plaza Serrano: {gdf_ent['dist_plaza_serrano'].mean():.2f} km")

### 6.2. Distancia a la Estación de Subte más Cercana
Para calcular distancias de forma masiva, lo ideal es:
1. Proyectar los datos a un sistema métrico (ej. **EPSG:22185** - Gauss-Kruger Buenos Aires).
2. Usar `sjoin_nearest` para encontrar la infraestructura más cercana.

In [ ]:
# Cargamos estaciones de subte
df_subtes = pd.read_csv("estaciones_de_subte.csv")

# La columna geometry viene en formato WKT string, la convertimos
df_subtes['geometry'] = df_subtes['geometry'].apply(wkt.loads)
# EPSG:4326 es un poco el default de las proyecciones (que no lean esto los profes de GIS)
gdf_subtes = gpd.GeoDataFrame(df_subtes, crs="EPSG:4326")

# Proyectamos ambos GDFs a metros (Gauss-Kruger faja 4)
# CRS: Sistema de Referencia de Coordenadas
gdf_ent_m = gdf_ent.to_crs("EPSG:22185")
gdf_subtes_m = gdf_subtes.to_crs("EPSG:22185")

# Buscamos la estación más cercana para cada propiedad
gdf_ent_m = gpd.sjoin_nearest(
    gdf_ent_m,
    gdf_subtes_m[['estacion', 'linea', 'geometry']],
    distance_col="dist_subte_metros",
    how="left"
)

print("Información de cercanía a subte:")
gdf_ent_m[['location_3', 'price', 'estacion', 'dist_subte_metros']].head()

## 7. Recuperación de Registros mediante Imputación Espacial

En el Paso 3, descartamos muchos registros por falta de coordenadas. Sin embargo, si el registro tiene información del barrio (`location_3`), podemos recuperar su posición aproximada utilizando la media espacial (centroide) del barrio.

In [ ]:
# Reiniciamos el dataframe para demostrar la recuperación
df_missing = df_ent.copy()

# Aplicamos los mismos filtros que en el paso 2
mask_filters = (
    df_missing["price"].notna() &
    (df_missing["currency_type"] == 'dolares') &
    df_missing["location_1"].isin(["Capital Federal", "Ciudad Autónoma de Buenos Aires", 'CABA']) &
    (df_missing["operation_type"] == 'venta') &
    df_missing["property_type"].isin(["departamento", "casa", "cochera"])
)
df_missing = df_missing.loc[mask_filters]

In [ ]:
# Identificamos registros sin lat/lon pero que SÍ tienen barrio (location_3)
registros_imputables = (df_missing["lat"].isna() | df_missing["lon"].isna()) & df_missing["location_3"].notna()
print(f"Registros candidatos a recuperación: {registros_imputables.sum()}")

In [ ]:
# Calculamos los centroides de cada barrio (media espacial del polígono)
gdf_barrios_idx = gdf_barrios.set_index("nombre")
centroids = gdf_barrios_idx.geometry.centroid

In [ ]:
# Convertimos los centroides en un GeoDataFrame para graficar
gdf_centroids = gpd.GeoDataFrame(
    geometry=centroids,
    crs=gdf_barrios.crs
)

fig, ax = plt.subplots(figsize=(10, 10))
gdf_barrios.plot(ax=ax, color='whitesmoke', edgecolor='gray', alpha=0.7)
gdf_centroids.plot(ax=ax, color='red', marker='x', markersize=50, label='Centroides')

for nombre, point in gdf_centroids.geometry.items():
    ax.text(point.x + 0.001, point.y + 0.001, nombre, fontsize=8, color='black')

plt.title("Centroides de los Barrios de CABA sobre el Mapa de Barrios")
plt.legend()
plt.show()

In [ ]:
# Mapeamos los centroides a los registros faltantes
df_imputed = df_missing[registros_imputables].copy()
df_imputed["lat"] = df_imputed["location_3"].map(lambda barrio: centroids[barrio].y if barrio in centroids.index else np.nan)
df_imputed["lon"] = df_imputed["location_3"].map(lambda barrio: centroids[barrio].x if barrio in centroids.index else np.nan)

In [ ]:
# Verificamos cuántos pudimos recuperar
recovered_count = df_imputed["lat"].notna().sum()
print(f"Registros recuperados exitosamente: {recovered_count}")

# Mostrar los primeros registros recuperados
df_imputed[["location_3", "lat", "lon"]].head()

## 8. Imputación de Barrios usando Coordenadas

Vamos a crear una nueva columna `location_3_imputada` calculada a partir de la latitud y longitud del registro respecto a los polígonos de los barrios de CABA. Compararemos esta nueva columna con la original `location_3`.

In [ ]:
# Reiniciamos el dataframe para asegurar que sea autocontenido
df_imputacion = df_ent.copy()

# Aplicamos los filtros básicos
mask_filters = (
    df_imputacion["price"].notna() &
    (df_imputacion["currency_type"] == 'dolares') &
    df_imputacion["location_1"].isin(["Capital Federal", "Ciudad Autónoma de Buenos Aires", 'CABA']) &
    (df_imputacion["operation_type"] == 'venta') &
    df_imputacion["property_type"].isin(["departamento", "casa", "cochera"])
)
df_imputacion = df_imputacion.loc[mask_filters]

In [ ]:
# Descartamos los registros que no tengan lat o lon
df_imputacion = df_imputacion.dropna(subset=["lat", "lon"])

# Convertimos a GeoDataFrame
gdf_imputacion = gpd.GeoDataFrame(
    df_imputacion,
    geometry=gpd.points_from_xy(df_imputacion["lon"], df_imputacion["lat"]),
    crs="EPSG:4326"
)

In [ ]:
# Realizamos el spatial join con gdf_barrios
gdf_imputacion = gpd.sjoin(gdf_imputacion, gdf_barrios[['nombre', 'geometry']], how='left', predicate='within')

# Renombramos la columna 'nombre' a 'location_3_imputada'
gdf_imputacion = gdf_imputacion.rename(columns={'nombre': 'location_3_imputada'})

In [ ]:
# Resumen de diferencias
# Alineamos los textos para hacer la comparacion justa
loc3 = gdf_imputacion['location_3'].astype(str).str.lower().str.strip()
loc3_imp = gdf_imputacion['location_3_imputada'].astype(str).str.lower().str.strip()

# Casos donde coinciden
coinciden = (loc3 == loc3_imp).sum()

# Casos donde location_3 era nulo y se imputó
nulos_imputados = (gdf_imputacion['location_3'].isna() & gdf_imputacion['location_3_imputada'].notna()).sum()

# Casos donde location_3 era distinto a la imputación (y no era nulo)
distintos = (gdf_imputacion['location_3'].notna() & (loc3 != loc3_imp)).sum()

# Casos donde no se pudo imputar (fuera de CABA)
no_imputados = gdf_imputacion['location_3_imputada'].isna().sum()

print(f"Total de registros con coordenadas: {len(gdf_imputacion)}")
print(f"Coinciden location_3 y la imputación: {coinciden}")
print(f"location_3 era nulo y se logró imputar: {nulos_imputados}")
print(f"location_3 era distinto a la imputación espacial: {distintos}")
print(f"Registros que no cayeron en ningún barrio (fuera de polígonos): {no_imputados}")


¿Teorias de porque da tan diferente?

## 9. Enfoque alternativo: Zona de Influencia del Subte (Buffers)

En lugar de calcular la distancia exacta, a veces es más útil categorizar las propiedades como "cerca" o "lejos" de la infraestructura. Definiremos una **zona de influencia de 500 metros** alrededor de cada estación utilizando buffers.

In [ ]:
# 1. Creamos los buffers de 500 metros (trabajando en metros EPSG:22185)
gdf_subtes_m['buffer_500'] = gdf_subtes_m.geometry.buffer(500)

# Creamos un GDF de polígonos para los buffers
gdf_buffers = gpd.GeoDataFrame(
    gdf_subtes_m[['estacion', 'linea']],
    geometry=gdf_subtes_m['buffer_500'],
    crs="EPSG:22185" # Sistema de coordenadas en metros
)

In [ ]:
# 2. Spatial Join Binario
# Disolvemos los buffers en un solo polígono para facilitar el chequeo binario
buffer_total = gdf_buffers.dissolve().geometry.iloc[0]

# Agregamos la columna binaria usando .within()
gdf_ent_m['cerca_subte'] = gdf_ent_m.geometry.within(buffer_total).astype(int)

In [ ]:
# 3. Visualización de los Buffers
fig, ax = plt.subplots(figsize=(10, 8))

# Capa 1: Barrios (proyectados)
gdf_barrios.to_crs("EPSG:22185").plot(ax=ax, color='whitesmoke', edgecolor='lightgray')

# Capa 2: Buffers
gdf_buffers.plot(ax=ax, color='red', alpha=0.2, edgecolor='red', label='Zona de Influencia (500m)')

# Capa 3: Estaciones
gdf_subtes_m.plot(ax=ax, color='red', markersize=10, marker='o', label='Estaciones de Subte')

plt.title("Zonas de Influencia de 500m alrededor de Estaciones de Subte")
plt.legend()
plt.show()

print(f"Total propiedades analizadas: {len(gdf_ent_m)}")
print(f"Propiedades dentro de la zona de influencia: {gdf_ent_m['cerca_subte'].sum()}")
print(f"Porcentaje de la oferta cerca del subte: {gdf_ent_m['cerca_subte'].mean()*100:.2f}%")

## 10. Visualización Final

Graficamos una muestra de las propiedades y las estaciones de subte sobre el mapa de barrios.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Capa 1: Barrios
gdf_barrios.plot(ax=ax, color='whitesmoke', edgecolor='gray', alpha=0.5)

# Capa 2: Propiedades (Muestra de 2000 para no saturar)
gdf_ent.sample(2000).plot(ax=ax, column='dist_plaza_serrano', cmap='viridis',
                          markersize=5, label='Propiedades', legend=True,
                          legend_kwds={'label': "Distancia a Plaza Serrano (km)"})

# Capa 3: Estaciones de Subte
gdf_subtes.plot(ax=ax, color='red', marker='D', markersize=20, label='Subte')

plt.title("Distribución de Inmuebles y Red de Subte en CABA")
plt.legend()
plt.show()